# Phase 6: Dataset Compatibility Analysis

Goals:
- dataset size and balance analysis
- overlap ratio and what it means
- data quality indicators
- Sample size adequacy for DA

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

# Path settings
ROOT = r'..'
BENCHMARK_DIR = os.path.join(ROOT, 'benchmark')
RESULTS_DIR = os.path.join(BENCHMARK_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*100)
print("PHASE 6: Dataset Compatibility Analysis")
print("="*100)
print(f"Results: {RESULTS_DIR}\n")

PHASE 6: Dataset Compatibility Analysis
Results: ..\benchmark\results



In [2]:
# Load data
print("Loading data...\n")

with open(os.path.join(RESULTS_DIR, 'preprocessed_data.pkl'), 'rb') as f:
    phase1_data = pickle.load(f)

with open(os.path.join(RESULTS_DIR, 'phase_01_overlap.json'), 'r') as f:
    overlap_data = json.load(f)

source_data = phase1_data['source_data']
target_data = phase1_data['target_data']
PROPERTIES = phase1_data['properties']
FOLD = phase1_data['fold']

print(f"Properties: {PROPERTIES}")
print(f"Fold: {FOLD}\n")

Loading data...

Properties: ['clearance', 'half_life', 'fu']
Fold: 0



In [3]:
# Dataset Compatibility Analysis
dataset_compatibility_results = {}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    dataset_compatibility_results[prop] = {}
    
    src_df = source_data[prop]
    tgt_df = target_data[prop]
    
    n_src = len(src_df)
    n_tgt = len(tgt_df)
    
    print(f"\n  Dataset sizes:")
    print(f"    Source: {n_src}")
    print(f"    Target: {n_tgt}")
    print(f"    Ratio (tgt/src): {n_tgt/n_src:.3f}")
    
    # Value statistics
    y_src = src_df['value'].values
    y_tgt = tgt_df['value'].values
    
    print(f"\n  Value distributions:")
    print(f"    Source: mean={np.mean(y_src):.4f}, std={np.std(y_src):.4f}, range=[{np.min(y_src):.4f}, {np.max(y_src):.4f}]")
    print(f"    Target: mean={np.mean(y_tgt):.4f}, std={np.std(y_tgt):.4f}, range=[{np.min(y_tgt):.4f}, {np.max(y_tgt):.4f}]")
    
    # Coefficient of variation
    cv_src = np.std(y_src) / (np.abs(np.mean(y_src)) + 1e-10)
    cv_tgt = np.std(y_tgt) / (np.abs(np.mean(y_tgt)) + 1e-10)
    print(f"    CV (source): {cv_src:.4f}")
    print(f"    CV (target): {cv_tgt:.4f}")
    
    # Skewness and kurtosis
    from scipy.stats import skew, kurtosis
    skew_src = skew(y_src)
    skew_tgt = skew(y_tgt)
    kurt_src = kurtosis(y_src)
    kurt_tgt = kurtosis(y_tgt)
    
    print(f"    Skewness (src): {skew_src:.4f}, (tgt): {skew_tgt:.4f}")
    print(f"    Kurtosis (src): {kurt_src:.4f}, (tgt): {kurt_tgt:.4f}")
    
    # Overlap analysis
    overlap_info = overlap_data.get(prop, {})
    n_overlap = overlap_info.get('overlap_count', 0)
    overlap_pct_src = overlap_info.get('overlap_pct_src', 0)
    overlap_pct_tgt = overlap_info.get('overlap_pct_tgt', 0)
    
    print(f"\n  Overlap metrics:")
    print(f"    Overlap molecules: {n_overlap}")
    print(f"    % of source: {overlap_pct_src:.1f}%")
    print(f"    % of target: {overlap_pct_tgt:.1f}%")
    
    # Data quality indicators
    print(f"\n  Data quality indicators:")
    
    # Check for duplicates within domain (after InChI dedup)
    unique_src = src_df['inchi_key'].nunique()
    unique_tgt = tgt_df['inchi_key'].nunique()
    
    dedup_ratio_src = unique_src / n_src
    dedup_ratio_tgt = unique_tgt / n_tgt
    
    print(f"    Unique molecules (src): {unique_src}/{n_src} ({100*dedup_ratio_src:.1f}%)")
    print(f"    Unique molecules (tgt): {unique_tgt}/{n_tgt} ({100*dedup_ratio_tgt:.1f}%)")
    
    # Sample size adequacy
    print(f"\n  Sample size adequacy for DA:")
    
    # Rule of thumb: at least 100 samples recommended for domain adaptation
    adequate_size_threshold = 100
    
    src_adequate = n_src >= adequate_size_threshold
    tgt_adequate = n_tgt >= adequate_size_threshold
    
    print(f"    Source >= {adequate_size_threshold}: {src_adequate} ({n_src} samples)")
    print(f"    Target >= {adequate_size_threshold}: {tgt_adequate} ({n_tgt} samples)")
    
    # Balance between domains
    size_ratio = min(n_src, n_tgt) / max(n_src, n_tgt)
    balanced = size_ratio > 0.3  # Both domains should have significant data
    
    print(f"    Size balance (ratio): {size_ratio:.3f}")
    print(f"    Balanced (ratio > 0.3): {balanced}")
    
    # Overall compatibility score (0-100)
    compatibility_score = 0
    
    # Size adequacy (30 points)
    if src_adequate and tgt_adequate:
        compatibility_score += 30
    elif src_adequate or tgt_adequate:
        compatibility_score += 15
    
    # Balance (20 points)
    if balanced:
        compatibility_score += 20
    elif size_ratio > 0.1:
        compatibility_score += 10
    
    # Overlap (20 points)
    if n_overlap > 10:
        compatibility_score += 20
    elif n_overlap > 0:
        compatibility_score += 10
    
    # Value distribution similarity (20 points)
    cv_ratio = cv_tgt / (cv_src + 1e-10)
    if 0.7 < cv_ratio < 1.3:
        compatibility_score += 20
    elif 0.5 < cv_ratio < 2.0:
        compatibility_score += 10
    
    # Deduplication quality (10 points)
    if dedup_ratio_src > 0.95 and dedup_ratio_tgt > 0.95:
        compatibility_score += 10
    
    print(f"\n  Compatibility score: {compatibility_score}/100")
    
    if compatibility_score >= 80:
        compat_level = "Excellent"
    elif compatibility_score >= 60:
        compat_level = "Good"
    elif compatibility_score >= 40:
        compat_level = "Moderate"
    elif compatibility_score >= 20:
        compat_level = "Poor"
    else:
        compat_level = "Very Poor"
    
    print(f"  Compatibility level: {compat_level}")
    
    dataset_compatibility_results[prop] = {
        'n_source': int(n_src),
        'n_target': int(n_tgt),
        'ratio_tgt_src': float(n_tgt / n_src),
        'src_mean': float(np.mean(y_src)),
        'src_std': float(np.std(y_src)),
        'tgt_mean': float(np.mean(y_tgt)),
        'tgt_std': float(np.std(y_tgt)),
        'cv_src': float(cv_src),
        'cv_tgt': float(cv_tgt),
        'skew_src': float(skew_src),
        'skew_tgt': float(skew_tgt),
        'kurt_src': float(kurt_src),
        'kurt_tgt': float(kurt_tgt),
        'n_overlap': int(n_overlap),
        'overlap_pct_src': float(overlap_pct_src),
        'overlap_pct_tgt': float(overlap_pct_tgt),
        'unique_src': int(unique_src),
        'unique_tgt': int(unique_tgt),
        'dedup_ratio_src': float(dedup_ratio_src),
        'dedup_ratio_tgt': float(dedup_ratio_tgt),
        'size_balance_ratio': float(size_ratio),
        'compatibility_score': int(compatibility_score),
        'compatibility_level': compat_level,
    }


Property: CLEARANCE

  Dataset sizes:
    Source: 14387
    Target: 1027
    Ratio (tgt/src): 0.071

  Value distributions:
    Source: mean=0.0000, std=0.9999, range=[-7.0635, 5.1515]
    Target: mean=0.0180, std=0.9959, range=[-4.6359, 3.7305]
    CV (source): 9999300953.1518
    CV (target): 55.2457
    Skewness (src): 0.1723, (tgt): -0.4682
    Kurtosis (src): 4.7981, (tgt): 1.9654

  Overlap metrics:
    Overlap molecules: 0
    % of source: 0.0%
    % of target: 0.0%

  Data quality indicators:
    Unique molecules (src): 13564/14387 (94.3%)
    Unique molecules (tgt): 1027/1027 (100.0%)

  Sample size adequacy for DA:
    Source >= 100: True (14387 samples)
    Target >= 100: True (1027 samples)
    Size balance (ratio): 0.071
    Balanced (ratio > 0.3): False

  Compatibility score: 30/100
  Compatibility level: Poor

Property: HALF_LIFE

  Dataset sizes:
    Source: 9204
    Target: 1164
    Ratio (tgt/src): 0.126

  Value distributions:
    Source: mean=0.0000, std=0.9999, r

In [4]:
# Summary Report
print(f"\n{'='*100}")
print("DATASET COMPATIBILITY SUMMARY")
print(f"{'='*100}\n")

summary_data = []

for prop in PROPERTIES:
    res = dataset_compatibility_results[prop]
    row = {
        'Property': prop,
        'Source_N': res['n_source'],
        'Target_N': res['n_target'],
        'Balance': f"{res['size_balance_ratio']:.3f}",
        'Overlap': f"{res['n_overlap']}",
        'Score': res['compatibility_score'],
        'Level': res['compatibility_level'],
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print()

print("Compatibility levels:")
print("  80-100: Excellent (ideal for domain adaptation)")
print("  60-79:  Good (suitable for domain adaptation)")
print("  40-59:  Moderate (partial transfer expected)")
print("  20-39:  Poor (challenging transfer)")
print("  0-19:   Very Poor (not recommended)")
print()


DATASET COMPATIBILITY SUMMARY

 Property  Source_N  Target_N Balance Overlap  Score    Level
clearance     14387      1027   0.071       0     30     Poor
half_life      9204      1164   0.126       0     40 Moderate
       fu      3783       353   0.093       0     40 Moderate

Compatibility levels:
  80-100: Excellent (ideal for domain adaptation)
  60-79:  Good (suitable for domain adaptation)
  40-59:  Moderate (partial transfer expected)
  20-39:  Poor (challenging transfer)
  0-19:   Very Poor (not recommended)



In [5]:
# Save results
with open(os.path.join(RESULTS_DIR, 'phase_06_dataset_compatibility.json'), 'w') as f:
    json.dump(dataset_compatibility_results, f, indent=2)

print("✓ Results saved to benchmark/results/")
print(f"  - phase_06_dataset_compatibility.json")
print("\nPhase 6 Complete! Ready for Phase 7: Transferability Profile")

✓ Results saved to benchmark/results/
  - phase_06_dataset_compatibility.json

Phase 6 Complete! Ready for Phase 7: Transferability Profile
